# TFM — Predicción de huella de carbono del consumo eléctrico (España)

## Notebook 01: Carga y preparación de datos (Wattnet)

**Objetivo**: construir una serie temporal limpia de **huella de carbono operacional global** para España (ES) con resolución horaria (agregada desde 15 minutos), a partir de los ficheros anuales 2022–2025.

**Salida**:
- `df_es`: serie temporal (timestamp index) con columna `y` (objetivo).
- Conjuntos `train`, `val`, `test` por división temporal:
  - Train: 2022–2023
  - Validation: 2024
  - Test: 2025

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 140)


## 1. Configuración

Definimos la ruta base donde se encuentran las carpetas anuales (2022–2025) del dataset de Wattnet.



In [2]:
from pathlib import Path

# Definir rutas y constantes
BASE_PATH = Path("/home/ubuntu/TFM")
DATA_RAW = BASE_PATH / "data_raw"

YEAR_FOLDERS = [
    "2022",
    "2023",
    "2024",
    "2025",
]

TARGET_FILE = Path("footprints") / "carbon_footprint_operational_global.csv"

COUNTRY_CODE = "ES"  # Spain
TIME_COL = "timestamp"


In [3]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parents[0]  
DATA_RAW = PROJECT_ROOT / "data_raw"

In [4]:
# Comprobar que los ficheros objetivo existen
paths = [DATA_RAW / y / TARGET_FILE for y in YEAR_FOLDERS]

missing = [str(p) for p in paths if not p.exists()]
if missing:
    raise FileNotFoundError(
        "No se han encontrado estos ficheros:\n" + "\n".join(missing) +
        "\n\nRevisa DATA_RAW, YEAR_FOLDERS o TARGET_FILE."
    )

paths


[PosixPath('/home/ubuntu/TFM/data_raw/2022/footprints/carbon_footprint_operational_global.csv'),
 PosixPath('/home/ubuntu/TFM/data_raw/2023/footprints/carbon_footprint_operational_global.csv'),
 PosixPath('/home/ubuntu/TFM/data_raw/2024/footprints/carbon_footprint_operational_global.csv'),
 PosixPath('/home/ubuntu/TFM/data_raw/2025/footprints/carbon_footprint_operational_global.csv')]

## 2. Lectura de ficheros y concatenación

Se cargan los datos de huella de carbono operacional global, que contienen información temporal con resolución original de 15 minutos para distintos países europeos. Posteriormente se agregarán a resolución horaria.

Cada CSV contiene una columna `timestamp` y una columna por país.  
Extraemos únicamente `timestamp` y la columna `ES` para reducir memoria y evitar errores posteriores.

Se selecciona España como caso de estudio principal debido a la disponibilidad de datos completos y a la diversidad de su mix energético.


In [5]:
def load_country_series(csv_path: Path, time_col: str, country_col: str) -> pd.DataFrame:
    """
    Lee un CSV de Wattnet con columnas: timestamp, AT, BA, ..., ES, ...
    Devuelve DataFrame con columnas: timestamp, y (float)
    """
    df = pd.read_csv(csv_path, usecols=[time_col, country_col])
    
    # Parseo de tiempo
    df[time_col] = pd.to_datetime(df[time_col], errors="raise", utc=True)
    
    # Renombrar objetivo
    df = df.rename(columns={country_col: "y"})
    
    # Asegurar tipo numérico
    df["y"] = pd.to_numeric(df["y"], errors="coerce")
    
    return df


In [6]:
dfs = []
for p in paths:
    df_part = load_country_series(p, TIME_COL, COUNTRY_CODE)
    dfs.append(df_part)

df_es = pd.concat(dfs, ignore_index=True)

df_es.head(), df_es.shape


(                  timestamp       y
 0 2022-01-01 00:00:00+00:00  120.79
 1 2022-01-01 00:15:00+00:00  120.79
 2 2022-01-01 00:30:00+00:00  120.79
 3 2022-01-01 00:45:00+00:00  120.80
 4 2022-01-01 01:00:00+00:00  120.04,
 (140256, 2))

## 3. Limpieza y formateo

- Ordenar por tiempo
- Establecer `timestamp` como índice
- Comprobar valores nulos en `y`


In [7]:
df_es = df_es.sort_values(TIME_COL)

# Duplicados exactos de timestamp no deberían existir, pero por si acaso los contamos y eliminamos
n_dupes = df_es.duplicated(subset=[TIME_COL]).sum()
print("Duplicados de timestamp:", n_dupes)

if n_dupes > 0:
    df_es = df_es.drop_duplicates(subset=[TIME_COL], keep="first")

df_es = df_es.set_index(TIME_COL)
df_es = df_es.sort_index()

df_es.info()


Duplicados de timestamp: 0
<class 'pandas.DataFrame'>
DatetimeIndex: 140256 entries, 2022-01-01 00:00:00+00:00 to 2025-12-31 23:45:00+00:00
Data columns (total 1 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   y       140244 non-null  float64
dtypes: float64(1)
memory usage: 2.1 MB


In [8]:
missing_y = df_es["y"].isna().sum()
total = len(df_es)
print(f"Nulos en y: {missing_y} ({missing_y/total:.4%})")

df_es.describe()


Nulos en y: 12 (0.0086%)


,y
count,140244.000000
mean,114.354448
std,48.763052
min,31.850000
25%,73.950000
50%,107.590000
75%,147.720000
max,281.850000


## 4. Comprobación de frecuencia temporal (15 min)

Convertimos la variable temporal al formato de fecha y hora (solo por si no lo está ya) y se establece como índice del dataset para facilitar el análisis de series temporales.
Verificamos que la separación entre timestamps sea 15 minutos.  
Si hay huecos, los detectaremos y los trataremos mediante reindexado e interpolación antes de agregar a resolución horaria.

In [9]:
deltas = df_es.index.to_series().diff().dropna()

# Delta más común
mode_delta = deltas.mode().iloc[0]
print("Delta más común:", mode_delta)

# Top 5 deltas por frecuencia
print(deltas.value_counts().head(5))

# Contar huecos (deltas mayores a 15 min)
expected = pd.Timedelta(minutes=15)
gaps = (deltas > expected).sum()
print("Número de saltos > 15 min:", gaps)


Delta más común: 0 days 00:15:00
timestamp
0 days 00:15:00    140255
Name: count, dtype: int64
Número de saltos > 15 min: 0


In [10]:
full_index = pd.date_range(
    start=df_es.index.min(),
    end=df_es.index.max(),
    freq="15min",
    tz="UTC"
)

df_es = df_es.reindex(full_index)
df_es.index.name = "timestamp"

print("Filas tras reindexado:", len(df_es))
print("Nulos tras reindexado:", df_es["y"].isna().sum())
df_es.head()


Filas tras reindexado: 140256
Nulos tras reindexado: 12


,y
timestamp,
2022-01-01 00:00:00+00:00,120.79
2022-01-01 00:15:00+00:00,120.79
2022-01-01 00:30:00+00:00,120.79
2022-01-01 00:45:00+00:00,120.80
2022-01-01 01:00:00+00:00,120.04


Reorganizamos la serie temporal a una frecuencia regular de 15 minutos, como medida de verificación (aunque los datos originales ya presentaban dicha resolución), garantizando la continuidad temporal necesaria antes de la agregación horaria.

In [11]:
train = df_es.loc["2022-01-01":"2023-12-31 23:59:59"]
val   = df_es.loc["2024-01-01":"2024-12-31 23:59:59"]
test  = df_es.loc["2025-01-01":"2025-12-31 23:59:59"]

print("Train:", train.shape, "NaNs:", train["y"].isna().sum())
print("Val:  ", val.shape,   "NaNs:", val["y"].isna().sum())
print("Test: ", test.shape,  "NaNs:", test["y"].isna().sum())


Train: (70080, 1) NaNs: 4
Val:   (35136, 1) NaNs: 3
Test:  (35040, 1) NaNs: 5


## 5. Imputación de valores faltantes y agregación horaria

Dado que el porcentaje de valores faltantes es muy bajo (≈0.009%), realizamos imputación mediante interpolación temporal para mantener una rejilla regular.

A continuación, agregamos la serie a **resolución horaria** calculando la media de los 4 registros de 15 minutos de cada hora. Esta agregación suaviza el ruido de alta frecuencia y reduce la dimensionalidad de la serie, facilitando el modelado posterior.


In [12]:
# Rolling de 4 valores y quedarnos con cada 4º registro (las horas en punto)
df_es["y_roll"] = df_es["y"].rolling(4).mean()

# Filtrar solo las horas en punto (minuto == 0)
df_es = df_es[df_es.index.minute == 0][["y_roll"]].rename(columns={"y_roll": "y"})

print("Filas tras agregación horaria:", len(df_es))
print("Nulos:", df_es["y"].isna().sum())
df_es.head(10)

Filas tras agregación horaria: 35064
Nulos: 11


,y
timestamp,
2022-01-01 00:00:00+00:00,NaN
2022-01-01 01:00:00+00:00,120.6050
2022-01-01 02:00:00+00:00,120.8400
2022-01-01 03:00:00+00:00,123.1475
2022-01-01 04:00:00+00:00,123.0475
2022-01-01 05:00:00+00:00,124.2675
2022-01-01 06:00:00+00:00,127.6975
2022-01-01 07:00:00+00:00,128.9975
2022-01-01 08:00:00+00:00,124.6075


In [19]:
# Ver si hay valores nulos
print("Valores nulos por columna:")
df_es.isnull().sum()

Valores nulos por columna:


y    11
dtype: int64

In [20]:
# Eliminar NaNs generados por el rolling en las primeras filas
df_es = df_es.dropna(subset=["y"])

print("Filas tras eliminar NaNs del rolling:", len(df_es))
print("NaNs restantes:", df_es["y"].isna().sum())

Filas tras eliminar NaNs del rolling: 35053
NaNs restantes: 0


Se eliminan las observaciones con valores faltantes generados por la media móvil al inicio de la serie, donde no se dispone de suficiente historia para completar la ventana de 4 registros. El número de filas afectadas es mínimo y no supone pérdida de información relevante.

In [21]:
# Comprobación: cargar datos originales a 15 min y verificar la agregación
# Cogemos una hora concreta y comparamos
hora_test = "2022-01-01 02:00:00+00:00"

# Datos originales: los 4 registros de 15 min anteriores a esa hora
# (01:15, 01:30, 01:45, 02:00)
df_original = pd.read_parquet("data_processed/es_carbon_footprint_operational_global_15min_2022_2025.parquet")

# Aquí usamos los datos que aún están en memoria antes del resample

mask = df_original.loc["2022-01-01 01:15:00":"2022-01-01 02:00:00", "y"]
print("Valores 15 min (01:15 a 02:00):")
print(mask)
print(f"\nMedia manual: {mask.mean():.4f}")
print(f"Valor en df_es resampleado para 02:00: {df_es.loc[hora_test, 'y']:.4f}")
print(f"¿Coinciden? {abs(mask.mean() - df_es.loc[hora_test, 'y']) < 1e-6}")

Valores 15 min (01:15 a 02:00):
timestamp
2022-01-01 01:15:00+00:00    120.04
2022-01-01 01:30:00+00:00    120.05
2022-01-01 01:45:00+00:00    120.05
2022-01-01 02:00:00+00:00    123.22
Name: y, dtype: float64

Media manual: 120.8400
Valor en df_es resampleado para 02:00: 120.8400
¿Coinciden? True


## 6. División temporal Train / Validation / Test

Se utiliza una división temporal para evitar fuga de información:

- **Train**: 2022–2023  
- **Validation**: 2024  
- **Test**: 2025  

Esta estrategia permite seleccionar hiperparámetros en validación y reportar resultados finales en test.
Los datos se encuentran ya a resolución horaria.


In [22]:
train = df_es.loc["2022-01-01":"2023-12-31 23:59:59"]
val   = df_es.loc["2024-01-01":"2024-12-31 23:59:59"]
test  = df_es.loc["2025-01-01":"2025-12-31 23:59:59"]

print("Train:", train.shape, "NaNs:", train["y"].isna().sum())
print("Val:  ", val.shape,   "NaNs:", val["y"].isna().sum())
print("Test: ", test.shape,  "NaNs:", test["y"].isna().sum())


Train: (17515, 1) NaNs: 0
Val:   (8782, 1) NaNs: 0
Test:  (8756, 1) NaNs: 0


## 7. Guardado de datos procesados (intermedio)

Guardamos en formato eficiente para acelerar el trabajo posterior (EDA/modelos).  
Formato recomendado: **Parquet**.


In [23]:
from pathlib import Path

OUT_DIR = Path("data_processed")
OUT_DIR.mkdir(exist_ok=True)

df_es.to_parquet(OUT_DIR / "es_carbon_footprint_operational_global_1h_2022_2025.parquet")
train.to_parquet(OUT_DIR / "train_2022_2023.parquet")
val.to_parquet(OUT_DIR / "val_2024.parquet")
test.to_parquet(OUT_DIR / "test_2025.parquet")

print("Guardado en:", OUT_DIR.resolve())
print(f"Serie completa: {len(df_es)} filas (resolución horaria)")
print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")


Guardado en: /home/ubuntu/TFM/notebooks/data_processed
Serie completa: 35053 filas (resolución horaria)
Train: 17515 | Val: 8782 | Test: 8756
